- Name: Yin Zhang
- Student ID: [Student ID removed]

In [ ]:
# Install required packages, if you haven't installed them
! pip install uv
! pip install torchinfo
! pip install transformers==4.57.6 --no-deps --no-cache-dir
! pip install huggingface_hub==0.36.2

In [ ]:
root_dir = "./"

In [ ]:
import math
import os
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from torchvision.transforms import v2


# A helper function to set the random seed for reproducibility across various libraries and frameworks
def set_seed(seed: int, deterministic: bool = True):
    # Copy from transformers.trainer_utilss.set_seed with some modifications
    """
    Helper function for reproducible behavior to set the seed in `random`, `numpy`, `torch` and/or `tf` (if installed).

    Args:
        seed (`int`):
            The seed to set.
        deterministic (`bool`, *optional*, defaults to `False`):
            Whether to use deterministic algorithms where available. Can slow down training.
    """

    import random

    import numpy as np
    import torch
    from transformers.utils.import_utils import is_torch_available


    random.seed(seed)
    np.random.seed(seed)
    # Python Hashing (affects dict/set ordering)

    os.environ["PYTHONHASHSEED"] = str(seed)

    if is_torch_available():
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

        if deterministic:
            os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
            os.environ["FLASH_ATTENTION_DETERMINISTIC"] = "1"
            torch.use_deterministic_algorithms(True)
    else:
        torch.manual_seed(seed)
    print(f"Global seed set to {seed}. Deterministic mode: {deterministic}")

set_seed(42)

# Load dataset

Tiny ImageNet-60k dataset is a
subset of Tiny ImageNet (https://huggingface.co/datasets/zh-plus/tiny-imagenet), consisting of
200 classes with a total of 60,000 training images and 10,000 test images.

### Load from Huggingface

In [ ]:
def collator_fn(batch):
    images = torch.stack([item['image'] for item in batch])
    labels = torch.tensor([item['label'] for item in batch], dtype=torch.long)
    return images, labels


def get_train_test_dataloader(batch_size=64, num_workers=4, transform=None):
    trainset = load_dataset('rudanie89/tiny_imagenet_60k', split="train")
    testset = load_dataset('rudanie89/tiny_imagenet_valid', split="valid")

    # Default test transform
    test_transform = v2.Compose([
        v2.ToImage(),
        v2.RGB(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    if transform is None:
        # Default train transform pipeline
        transform = v2.Compose([
            v2.ToImage(),
            v2.RGB(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    trainset.set_transform(transform)
    testset.set_transform(test_transform)

    trainloader = torch.utils.data.DataLoader(
        trainset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
        collate_fn=collator_fn
    )
    testloader = torch.utils.data.DataLoader(
        testset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
        collate_fn=collator_fn
    )

    return trainloader, testloader

In [ ]:
def imshow(img):
    """Display image with appropriate colormap."""
    cmap = None
    if len(img.shape) == 2:
        img = img.unsqueeze(0) # Add a channel dimension for grayscale images (H, W) -> (1, H, W)
    if img.shape[0] < 3:
        cmap = "gray"
    # Permute the image tensor from (C, H, W) to (H, W, C), then cast to numpy array for displaying with matplotlib
    plt.imshow(img.permute(1, 2, 0).numpy(), cmap=cmap)


def visualize_data(images, categories, class_names, images_per_row=8):
        """Visualize a batch of images with their class labels."""
        n_images = len(images)
        n_rows = math.ceil(float(n_images) / images_per_row)
        fig = plt.figure(figsize=(1.5 * images_per_row, 1.5 * n_rows))
        fig.patch.set_facecolor('white')

        for i in range(n_images):
            plt.subplot(n_rows, images_per_row, i + 1)
            plt.xticks([])
            plt.yticks([])
            imshow(images[i]) # Image is a PyTorch tensor
            class_index = categories[i]
            plt.xlabel(class_names[class_index])

        plt.show()

In [ ]:
"""
NOTE: THIS BLOCK IS FOR VISUALIZATION PURPOSE ONLY
"""

visualization_transform = v2.Compose([
    v2.ToImage(),
    v2.RGB(),
    v2.ToDtype(torch.float32, scale=True),
])

batch_size = 64

train_loader, _ = get_train_test_dataloader(
    batch_size=batch_size,
    num_workers=0,
    transform=visualization_transform
)

class_id = train_loader.dataset.info.features['label'].names # Get the mapping from label indices to class IDs (e.g., 0 -> 'n01443537')

label_dict = {}
with open(root_dir +"labels.txt", "r") as f: # Load the provided labels.txt that maps from class IDs to human-readable labels (e.g., 'n01443537' -> 'goldfish')
    for line in tqdm(f):
        id, label = line.split("\t")
        label_dict[id.strip()] = label.split(",")[0].strip() # Use the first label if there are multiple labels for a class

train_images, train_labels = next(iter(train_loader))
visualize_data(
    images=train_images,
    categories=[class_id[label] for label in train_labels], class_names=label_dict,
    images_per_row=16,
)

In [ ]:
batch_size = 64
transform = None # Define your own transform pipeline, otherwise it will default to the transform inside `get_train_test_dataloader`
train_loader, test_loader = get_train_test_dataloader(batch_size=batch_size, num_workers=0, transform=transform)

# Task 1 - Baseline CNN model (4 marks)

In [ ]:
import math
import os
import random
from tqdm.auto import tqdm
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
sample_images, _ = next(iter(train_loader))
input_shape = sample_images.shape[1:]
print(f"Input image shape: {input_shape}")

In [ ]:
import torch
import torch.nn as nn
from torchinfo import summary
class BaseCNN(nn.Module):
    def __init__(self, num_classes=200):
        super(BaseCNN, self).__init__()
        self.features = nn.Sequential(
            # Conv1: 3->16, kernel 7x7, stride 1, padding 0
            nn.Conv2d(3, 16, kernel_size=7, stride=1, padding=0),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 58 -> 29

            # Conv2: 16->32, kernel 5x5, stride 1, padding 2
            nn.Conv2d(16, 32, kernel_size=5, stride=1, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 29 -> 14

            # Conv3: 32->32, kernel 5x5, stride 1, padding 2
            nn.Conv2d(32, 32, kernel_size=5, stride=1, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 14 -> 7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 196),  # 32*7*7 = 1568
            nn.ReLU(inplace=True),
            nn.Linear(196, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [ ]:
device = device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BaseCNN(num_classes=200).to(device)

In [ ]:
print("\nModel Architecture")
summary(model, input_size=(1, 3, 64, 64), device=str(device))

In [ ]:
# Define the loss function, optimizer, and learning rate scheduler
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

# Set training epochs and initialise the best test accuracy
num_epochs = 20
best_test_acc = 0.0

# Lists used to store loss and accuracy values for plotting later
train_losses, test_losses = [], []
train_accs, test_accs = [], []

# Compute top-1 accuracy for one batch
def compute_accuracy(outputs, labels):
    _, preds = torch.max(outputs, 1)
    return (preds == labels).sum().item() / len(labels)

# Evaluate the model on a given data loader without updating weights
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    running_acc = 0.0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            # Move images and labels to the selected device
            images, labels = images.to(device), labels.to(device)

            # Forward pass and loss calculation
            outputs = model(images)
            loss = criterion(outputs, labels)

            # Accumulate loss and accuracy
            running_loss += loss.item() * images.size(0)
            running_acc += compute_accuracy(outputs, labels) * images.size(0)
            total += images.size(0)

    return running_loss / total, running_acc / total

In [ ]:
# Train the model for the selected number of epochs
for epoch in range(1, num_epochs + 1):
    print(f"Starting Epoch {epoch}/{num_epochs}...", flush=True)

    model.train()
    train_loss = 0.0
    train_acc = 0.0
    total_train = 0

    # Training loop over all batches
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        # Forward pass, loss calculation, and parameter update
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # Accumulate training loss and accuracy
        train_loss += loss.item() * images.size(0)
        train_acc += compute_accuracy(outputs, labels) * images.size(0)
        total_train += images.size(0)

    # Calculate average training loss and accuracy for this epoch
    epoch_train_loss = train_loss / total_train
    epoch_train_acc = train_acc / total_train

    # Evaluate the model on the test set after each epoch
    print("Evaluating on test set...", flush=True)
    epoch_test_loss, epoch_test_acc = evaluate(model, test_loader, criterion, device)

    # Store values for plotting curves later
    train_losses.append(epoch_train_loss)
    train_accs.append(epoch_train_acc)
    test_losses.append(epoch_test_loss)
    test_accs.append(epoch_test_acc)

    # Keep track of the best test accuracy
    if epoch_test_acc > best_test_acc:
        best_test_acc = epoch_test_acc

    # Print a short summary for the current epoch
    print(
        f"Epoch {epoch:02d}/{num_epochs} finished | "
        f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4f} | "
        f"Test Loss: {epoch_test_loss:.4f} | Test Acc: {epoch_test_acc:.4f} | "
        f"Best Test Acc: {best_test_acc:.4f}",
        flush=True
    )

print(f"\nBest test accuracy: {best_test_acc:.4f}", flush=True)

In [ ]:
# Plot training/testing loss and accuracy curves
def plot_curve(train_losses, test_losses, train_accs, test_accs):

    # Plot loss curves
    plt.figure(figsize=(8, 5))
    plt.plot(train_losses, 'b-o', label='Train Loss')
    plt.plot(test_losses, 'r-s', label='Test Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Test Loss')
    plt.legend()
    plt.grid(True)
    plt.show()

    # Plot accuracy curves
    plt.figure(figsize=(8, 5))
    plt.plot(train_accs, 'b-o', label='Train Accuracy')
    plt.plot(test_accs, 'r-s', label='Test Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.title('Training and Test Accuracy')
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
# Draw the loss and accuracy curves after training
plot_curve(train_losses, test_losses, train_accs, test_accs)

# Task 2 - Improve the baseline model from Task 1 (7 marks)

In [ ]:
# Train and evaluate a given model class with selected parameters and data transform
def train_and_evaluate(model_class, model_param, train_transform, num_epochs=20):
    # Create training and testing dataloaders
    train_loader, test_loader = get_train_test_dataloader(
        batch_size=256, num_workers=0,
        transform=train_transform)

    # Set training hyperparameters
    lr=0.001
    weight_decay=0

    # Build the model and move it to the selected device
    model = model_class(**model_param).to(device)

    # Define loss function, optimizer, and learning rate scheduler
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

    # Initialise best accuracy and lists for recording results
    best_acc = 0.0
    best_model = None
    train_losses, test_losses = [], []
    train_accs, test_accs = [], []

    # Main training loop
    for epoch in range(1, num_epochs+1):
        model.train()
        running_loss, running_acc, total = 0.0, 0.0, 0

        # Train over all batches
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            # Forward pass, loss calculation, and parameter update
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            # Accumulate training loss and correct predictions
            batch_size_curr = images.size(0)
            running_loss += loss.item() * batch_size_curr
            running_acc += (outputs.argmax(1) == labels).sum().item()
            total += batch_size_curr

        # Calculate average training loss and accuracy
        epoch_train_loss = running_loss / total
        epoch_train_acc = running_acc / total

        # Evaluate the model on the test set
        model.eval()
        test_loss, test_acc = 0.0, 0.0
        total_test = 0

        # No gradient is needed during evaluation
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)

                # Accumulate testing loss and correct predictions
                test_loss += loss.item() * images.size(0)
                test_acc += (outputs.argmax(1) == labels).sum().item()
                total_test += images.size(0)

        # Calculate average testing loss and accuracy
        epoch_test_loss = test_loss / total_test
        epoch_test_acc = test_acc / total_test

        # Store values for plotting curves later
        train_losses.append(epoch_train_loss)
        train_accs.append(epoch_train_acc)
        test_losses.append(epoch_test_loss)
        test_accs.append(epoch_test_acc)

        # Print test performance for this epoch
        print(f"Epoch {epoch:2d} | Test Loss {epoch_test_loss:.4f} Acc {epoch_test_acc:.4f}")

        # Keep track of the model with the best test accuracy
        if epoch_test_acc > best_acc:
            best_acc = epoch_test_acc
            best_model = model

        # Update learning rate according to scheduler
        scheduler.step()

    # Return best accuracy, history values, and best model
    return best_acc, (train_losses, train_accs, test_losses, test_accs), best_model

## 1.Data augmentation

In [ ]:
from torchvision.transforms import v2 as transforms

# Define two data augmentation settings for comparison
aug_configs = {
    # Aug_1 uses simple augmentation: horizontal flip and random crop
    'aug_1': transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomCrop(64, padding=4),

        # Convert image to tensor format and normalise pixel values
        transforms.ToImage(), transforms.RGB(),
        transforms.ToDtype(torch.float32, scale=True),
        transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ]),

    # Aug_2 uses stronger augmentation with affine transformation and colour changes
    'aug_2': transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomAffine(degrees=15, translate=(0.1,0.1), scale=(0.9,1.1)),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.RandomCrop(64, padding=4),

        # Convert image to tensor format and apply ImageNet-style normalisation
        transforms.ToImage(), transforms.RGB(),
        transforms.ToDtype(torch.float32, scale=True),
        transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ])
}

In [ ]:
# Store the results of different data augmentation settings
aug_results = {}

# Train and evaluate the baseline CNN with each augmentation setting
for name, train_tf in aug_configs.items():
    print(f"\nData Augmentation: {name}:\n")

    best_acc, history, best_model = train_and_evaluate(
        BaseCNN, {'num_classes': 200}, train_tf
    )

    # Save the best accuracy, training history, and model for later comparison
    aug_results[name] = {
        'best_acc': best_acc,
        'history': history,
        'model': best_model
    }

In [ ]:
# Print and plot the results for each augmentation setting
for name, train_tf in aug_configs.items():
    print(name)
    print('best_acc', aug_results[name]['best_acc'])

    # Get the recorded loss and accuracy values
    (train_losses, train_accs, test_losses, test_accs) = aug_results[name]['history']

    # Plot training/testing loss and accuracy curves
    plot_curve(train_losses, test_losses, train_accs, test_accs)

## 2.Norm and Drop

In [ ]:
# CNN model with Batch Normalisation and Dropout added to the baseline structure
class NormDropCNN(nn.Module):
    def __init__(self, num_classes=200, dropout_rate=0.5):
        super().__init__()

        # First convolution block: Conv -> ReLU -> BatchNorm -> MaxPool
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 16, 7, 1, 0), nn.ReLU(),
            nn.BatchNorm2d(16),
            nn.MaxPool2d(2, 2)
        )

        # Second convolution block
        self.conv2 = nn.Sequential(
            nn.Conv2d(16, 32, 5, 1, 2), nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(2, 2)
        )

        # Third convolution block
        self.conv3 = nn.Sequential(
            nn.Conv2d(32, 32, 5, 1, 2), nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(2, 2)
        )

        # Classifier with Dropout before the final output layer
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 196), nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(196, num_classes)
        )

    # Forward pass through convolution blocks and classifier
    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.classifier(x)
        return x


# Train and evaluate the BatchNorm + Dropout model using Aug_1
norm_results = {}
train_tf = aug_configs['aug_1']

best_acc, hist, best_model = train_and_evaluate(
    NormDropCNN,
    {'num_classes': 200},
    train_tf,
    num_epochs=20
)

# Store results for reporting and plotting
norm_results = {
    'best_acc': best_acc,
    'history': hist,
    'model': best_model
}

print("\nNorm/Dropout Results")
print(f"best test acc: {best_acc:.4f}")

# Plot loss and accuracy curves
(train_losses, train_accs, test_losses, test_accs) = norm_results['history']
plot_curve(train_losses, test_losses, train_accs, test_accs)

### 3.Modified Kernel

In [ ]:
# CNN model used to test different kernel sizes and channel numbers
class ModifiedKernelCNN(nn.Module):
    def __init__(self, num_classes=200, conv_channels=[32,64,64], kernel_sizes=[5,3,3]):
        super().__init__()

        # Read channel and kernel settings
        c1, c2, c3 = conv_channels
        k1, k2, k3 = kernel_sizes

        # Padding keeps the feature map size stable for odd kernel sizes
        p1 = (k1-1)//2 if k1%2 else 0
        p2 = (k2-1)//2 if k2%2 else 0
        p3 = (k3-1)//2 if k3%2 else 0

        # Feature extractor with three convolution blocks
        self.features = nn.Sequential(
            nn.Conv2d(3, c1, k1, 1, p1), nn.ReLU(), nn.MaxPool2d(2,2),
            nn.Conv2d(c1, c2, k2, 1, p2), nn.ReLU(), nn.MaxPool2d(2,2),
            nn.Conv2d(c2, c3, k3, 1, p3), nn.ReLU(), nn.MaxPool2d(2,2),
        )

        # Use a dummy input to calculate the flattened feature size automatically
        with torch.no_grad():
            dummy = torch.zeros(1,3,64,64)
            out = self.features(dummy)
            flat_dim = out.view(1,-1).size(1)

        # Classifier part
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat_dim, 196), nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(196, num_classes)
        )

    # Forward pass
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


# Different kernel/channel configurations for comparison
kernel_configs = [
    ('small_kernel', {'conv_channels':[16,32,32], 'kernel_sizes':[3,3,3]}),
    ('wide_channels', {'conv_channels':[32,64,64], 'kernel_sizes':[7,5,5]}),
    ('deep_wide',   {'conv_channels':[32,64,128], 'kernel_sizes':[5,3,3]}),
]

# Train and evaluate each configuration
kernel_results = {}
for name, kwargs in kernel_configs:
    print(f"\nKernel/Channel:: {name}")
    train_tf = aug_configs['aug_1']

    best_acc, hist, best_model = train_and_evaluate(
        ModifiedKernelCNN,
        {'num_classes':200, **kwargs},
        train_tf,
        num_epochs=20
    )

    # Store results for comparison and plotting
    kernel_results[name] = {
        'best_acc': best_acc,
        'history': hist,
        'model': best_model
    }

In [ ]:
# Print and plot the results for each kernel/channel configuration
print("\nKernel/Channel Results")

for name, res in kernel_results.items():
    # Print the best test accuracy for this configuration
    print(f"{name:20s} best test acc: {res['best_acc']:.4f}")

    # Get the recorded loss and accuracy values
    (train_losses, train_accs, test_losses, test_accs) = kernel_results[name]['history']

    # Plot training/testing loss and accuracy curves
    plot_curve(train_losses, test_losses, train_accs, test_accs)

### 4.Pool type

In [ ]:
# CNN model used to test extra depth and different pooling strategies
class DeeperPoolCNN(nn.Module):
    def __init__(self, num_classes=200, add_layers=2, pool_type='avg'):
        super().__init__()

        layers = []

        # Initial convolution blocks
        layers.extend([
            nn.Conv2d(3,16,3,1,1), nn.ReLU(), nn.BatchNorm2d(16), nn.MaxPool2d(2,2),
            nn.Conv2d(16,32,3,1,1), nn.ReLU(), nn.BatchNorm2d(32), nn.MaxPool2d(2,2),
        ])

        # Add extra convolution layers to increase model depth
        in_ch = 32
        for i in range(add_layers):
            out_ch = 64 if i==0 else 64
            layers.extend([
                nn.Conv2d(in_ch, out_ch, 3,1,1), nn.ReLU(), nn.BatchNorm2d(out_ch)
            ])
            in_ch = out_ch

        # Select the final pooling strategy
        if pool_type == 'avg':
            layers.append(nn.AdaptiveAvgPool2d((1,1)))
        else:
            layers.append(nn.MaxPool2d(2,2))
            layers.append(nn.AdaptiveAvgPool2d((1,1)))

        self.features = nn.Sequential(*layers)

        # Classifier after feature extraction
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_ch, 196), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(196, num_classes)
        )

    # Forward pass through feature extractor and classifier
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [ ]:
# Store results for different pooling strategies
depth_results = {}

# Compare average pooling and max pooling in the deeper CNN
for pool_type in ['avg', 'max']:
    print(f"\nDepth/Pool: ", pool_type)

    # Use the better augmentation setting from the previous experiment
    train_tf = aug_configs['aug_1']

    # Train and evaluate the model with the selected pooling type
    best_acc, hist, best_model = train_and_evaluate(
        DeeperPoolCNN,
        {'num_classes': 200, 'add_layers': 2, 'pool_type': pool_type},
        train_tf,
        num_epochs=20
    )

    # Save the result for later comparison and plotting
    depth_results[pool_type] = {
        'best_acc': best_acc,
        'history': hist,
        'model': best_model
    }

In [ ]:
# Print and plot the results for each pooling strategy
print("\nPool Results")

for name, res in depth_results.items():
    # Print the best test accuracy for this pooling type
    print(f"{name:20s} best test acc: {res['best_acc']:.4f}")

    # Get the recorded loss and accuracy values
    (train_losses, train_accs, test_losses, test_accs) = depth_results[name]['history']

    # Plot training/testing loss and accuracy curves
    plot_curve(train_losses, test_losses, train_accs, test_accs)

### 5.ResNet

In [ ]:
# Basic residual block with two convolution layers and a shortcut connection
class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()

        # Main path: Conv -> BN -> ReLU -> Conv -> BN
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride, 1)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1)
        self.bn2 = nn.BatchNorm2d(out_ch)

        # Shortcut path; use 1x1 convolution when size or channel number changes
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride, 0),
                nn.BatchNorm2d(out_ch)
            )

    # Forward pass with residual addition
    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = self.relu(out)
        return out


# ResNet-style CNN built manually using residual blocks
class ResNetCNN(nn.Module):
    def __init__(self, num_classes=200):
        super().__init__()

        # Initial convolution layer
        self.conv1 = nn.Conv2d(3, 64, 3, 1, 1)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU()

        # Three residual stages with increasing channel numbers
        self.layer1 = self._make_layer(64, 64, 2, stride=1)
        self.layer2 = self._make_layer(64, 128, 2, stride=2)
        self.layer3 = self._make_layer(128, 256, 2, stride=2)

        # Global average pooling and final classifier
        self.avgpool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(256, num_classes)

    # Helper function to build a residual stage
    def _make_layer(self, in_ch, out_ch, blocks, stride):
        layers = [ResidualBlock(in_ch, out_ch, stride)]
        for _ in range(1, blocks):
            layers.append(ResidualBlock(out_ch, out_ch, 1))
        return nn.Sequential(*layers)

    # Forward pass through the ResNet-style model
    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x


print("\n===== ResNet Module =====")

# Use Aug_1 for training the residual model
train_tf = aug_configs['aug_1']

# Train and evaluate the ResNet-style model
best_acc, hist, best_model_resent = train_and_evaluate(
    ResNetCNN,
    {'num_classes':200},
    train_tf,
    num_epochs=20
)

print(f"ResNet best test acc: {best_acc:.4f}")

In [ ]:
(train_losses, train_accs, test_losses, test_accs) = hist
plot_curve(train_losses,test_losses,train_accs,test_accs)


### 6.InceptionNet

In [ ]:
# Inception module with four branches for multi-scale feature extraction
class InceptionModule(nn.Module):
    def __init__(self, in_ch, out1, out3_red, out3, out5_red, out5, pool_proj):
        super().__init__()

        # 1x1 convolution branch
        self.branch1 = nn.Conv2d(in_ch, out1, 1)

        # 1x1 reduction followed by 3x3 convolution
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_ch, out3_red, 1),
            nn.Conv2d(out3_red, out3, 3, padding=1)
        )

        # 1x1 reduction followed by 5x5 convolution
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_ch, out5_red, 1),
            nn.Conv2d(out5_red, out5, 5, padding=2)
        )

        # Max pooling branch followed by 1x1 projection
        self.branch4 = nn.Sequential(
            nn.MaxPool2d(3, stride=1, padding=1),
            nn.Conv2d(in_ch, pool_proj, 1)
        )

    # Concatenate outputs from all branches along the channel dimension
    def forward(self, x):
        return torch.cat([self.branch1(x), self.branch2(x), self.branch3(x), self.branch4(x)], 1)


# CNN model using two Inception modules
class InceptionCNN(nn.Module):
    def __init__(self, num_classes=200):
        super().__init__()

        # Initial convolution and pooling layers
        self.conv1 = nn.Conv2d(3, 64, 7, 2, 3)   # 64x64 -> 32x32
        self.maxpool = nn.MaxPool2d(3, 2, 1)

        # Two Inception modules with increasing channel numbers
        self.inception1 = InceptionModule(64, 32, 32, 64, 16, 32, 16)   # output channels: 144
        self.inception2 = InceptionModule(144, 64, 64, 128, 32, 64, 32) # output channels: 288

        # Global average pooling and final classifier
        self.avgpool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(288, num_classes)

    # Forward pass through the Inception-based model
    def forward(self, x):
        x = self.conv1(x)
        x = self.maxpool(x)
        x = self.inception1(x)
        x = self.inception2(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x


# Select GPU if available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("\nInception Module")

# Use Aug_1 for training the Inception model
train_tf = aug_configs['aug_1']

# Create and train the Inception model
inception_model = InceptionCNN(num_classes=200).to(device)
best_acc, hist, best_model_inception = train_and_evaluate(
    InceptionCNN,
    {'num_classes':200},
    train_tf,
    num_epochs=20
)

print(f"Inception best test acc: {best_acc:.4f}")

In [ ]:
(train_losses, train_accs, test_losses, test_accs) = hist
plot_curve(train_losses,test_losses,train_accs,test_accs)

# Task 3 - Find the best model (5 marks)

In [ ]:
# Basic residual block with two convolution layers and a shortcut connection
class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()

        # Main path: Conv -> BN -> ReLU -> Conv -> BN
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride, 1)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1)
        self.bn2 = nn.BatchNorm2d(out_ch)

        # Shortcut path; use 1x1 convolution when size or channels change
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride, 0),
                nn.BatchNorm2d(out_ch)
            )

    # Forward pass with residual addition
    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = self.relu(out)
        return out


# ResNet-style CNN used as the final best model
class ResNetCNN(nn.Module):
    def __init__(self, num_classes=200):
        super().__init__()

        # Initial convolution layer
        self.conv1 = nn.Conv2d(3, 64, 3, 1, 1)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU()

        # Three residual stages
        self.layer1 = self._make_layer(64, 64, 2, stride=1)
        self.layer2 = self._make_layer(64, 128, 2, stride=2)
        self.layer3 = self._make_layer(128, 256, 2, stride=2)

        # Global average pooling and final classifier
        self.avgpool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(256, num_classes)

    # Helper function to build one residual stage
    def _make_layer(self, in_ch, out_ch, blocks, stride):
        layers = [ResidualBlock(in_ch, out_ch, stride)]
        for _ in range(1, blocks):
            layers.append(ResidualBlock(out_ch, out_ch, 1))
        return nn.Sequential(*layers)

    # Forward pass through the model
    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x


print("\n===== ResNet Module =====")

# Use Aug_1 for the final ResNet training
train_tf = aug_configs['aug_1']

# Train and evaluate the final ResNet model
final_best_acc, final_hist, final_best_model = train_and_evaluate(
   ResNetCNN, {'num_classes':200}, train_tf, num_epochs=30)

print(f"Final best model test acc: {final_best_acc:.4f}")

# Save the final best model checkpoint
torch.save(final_best_model.state_dict(), root_dir+"final_best_model.pth")

# Plot training/testing loss and accuracy curves
(train_losses, train_accs, test_losses, test_accs) = final_hist
plot_curve(train_losses, test_losses, train_accs, test_accs)

In [ ]:
print("\n===== Model Architecture =====")
summary(ResNetCNN(200), input_size=(1, 3, 64, 64), device=str(device))

#### Load checkpoint model and evaluate

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride, 1)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride, 0),
                nn.BatchNorm2d(out_ch)
            )

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = self.relu(out)
        return out


class ResNetCNN(nn.Module):
    def __init__(self, num_classes=200):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, 3, 1, 1)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU()
        self.layer1 = self._make_layer(64, 64, 2, stride=1)
        self.layer2 = self._make_layer(64, 128, 2, stride=2)
        self.layer3 = self._make_layer(128, 256, 2, stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(256, num_classes)

    def _make_layer(self, in_ch, out_ch, blocks, stride):
        layers = [ResidualBlock(in_ch, out_ch, stride)]
        for _ in range(1, blocks):
            layers.append(ResidualBlock(out_ch, out_ch, 1))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x


# Load the saved Task 3 checkpoint
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint_path = root_dir + "final_best_model.pth"

state_dict = torch.load(checkpoint_path, map_location=device)

# Rebuild the model structure and load the saved weights
final_model = ResNetCNN()
final_model.load_state_dict(state_dict)

# Task 4 - Explore image classification (4 marks)

In [ ]:
# Squeeze-and-Excitation block for channel attention
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(channels, channels // reduction),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        # Generate channel-wise weights and apply them to the feature map
        scale = self.fc(x).view(x.size(0), x.size(1), 1, 1)
        return x * scale


# DropBlock randomly removes local regions from feature maps during training
class DropBlock(nn.Module):
    def __init__(self, block_size=7, keep_prob=0.9):
        super().__init__()
        self.block_size = block_size
        self.keep_prob = keep_prob

    def forward(self, x):
        # Do not apply DropBlock during evaluation
        if not self.training or self.keep_prob == 1.0:
            return x

        gamma = (1 - self.keep_prob) / (self.block_size ** 2)
        mask = (torch.rand(x.shape[0], 1, x.shape[2], x.shape[3]) < gamma).to(x.device)
        mask = mask.float()

        # Expand dropped points into square blocks
        mask = torch.nn.functional.max_pool2d(
            mask, kernel_size=self.block_size, stride=1, padding=self.block_size//2
        )

        mask = 1 - mask
        x = x * mask

        # Rescale output to keep feature magnitude stable
        return x * (mask.numel() / mask.sum())


# DropPath randomly drops an entire residual path during training
def drop_path(x, drop_prob=0., training=False):
    if drop_prob == 0. or not training:
        return x

    keep_prob = 1 - drop_prob
    shape = (x.shape[0],) + (1,) * (x.ndim - 1)

    random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
    random_tensor.floor_()

    return x.div(keep_prob) * random_tensor


# Module wrapper for DropPath
class DropPath(nn.Module):
    def __init__(self, drop_prob=None):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        return drop_path(x, self.drop_prob, self.training)


# Pre-activation residual block with SE, DropBlock, and DropPath
class PreActResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, drop_rate=0.0, use_se=True):
        super().__init__()

        # Pre-activation design: BN and ReLU are placed before convolution
        self.bn1 = nn.BatchNorm2d(in_ch)
        self.relu = nn.ReLU(inplace=True)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride, 1)

        self.bn2 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1)

        # Optional attention and regularisation modules
        self.se = SEBlock(out_ch) if use_se else nn.Identity()
        self.dropblock = DropBlock(block_size=5, keep_prob=0.9)
        self.drop_path = DropPath(drop_rate)

        # Shortcut connection for channel or size change
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Conv2d(in_ch, out_ch, 1, stride, 0)

    def forward(self, x):
        identity = self.shortcut(x)

        out = self.relu(self.bn1(x))
        out = self.conv1(out)

        out = self.relu(self.bn2(out))
        out = self.conv2(out)

        out = self.se(out)
        out = self.dropblock(out)
        out = self.drop_path(out)

        out += identity
        return out


# Improved ResNet model for Task 4
class ImprovedResNet(nn.Module):
    def __init__(self, num_classes=200, width_factor=1, stage_blocks=[2,2,2,2], use_se=True, drop_rate=0.1):
        super().__init__()

        # Stem layer
        self.conv1 = nn.Conv2d(3, int(64*width_factor), 3, 1, 1)
        self.bn1 = nn.BatchNorm2d(int(64*width_factor))
        self.relu = nn.ReLU(inplace=True)

        # Four residual stages with increasing channel numbers
        self.layer1 = self._make_layer(int(64*width_factor), int(64*width_factor), stage_blocks[0], stride=1, use_se=use_se, drop_rate=drop_rate)
        self.layer2 = self._make_layer(int(64*width_factor), int(128*width_factor), stage_blocks[1], stride=2, use_se=use_se, drop_rate=drop_rate)
        self.layer3 = self._make_layer(int(128*width_factor), int(256*width_factor), stage_blocks[2], stride=2, use_se=use_se, drop_rate=drop_rate)
        self.layer4 = self._make_layer(int(256*width_factor), int(512*width_factor), stage_blocks[3], stride=2, use_se=use_se, drop_rate=drop_rate)

        # Final classifier
        self.avgpool = nn.AdaptiveAvgPool2d((1,1))
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(int(512*width_factor), num_classes)

    # Helper function to build each residual stage
    def _make_layer(self, in_ch, out_ch, blocks, stride, use_se, drop_rate):
        layers = []
        layers.append(PreActResBlock(in_ch, out_ch, stride, drop_rate, use_se))

        for _ in range(1, blocks):
            layers.append(PreActResBlock(out_ch, out_ch, 1, drop_rate, use_se))

        return nn.Sequential(*layers)

    # Forward pass through the improved model
    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)

        x = self.dropout(x)
        x = self.fc(x)

        return x

In [ ]:
print("\n===== ImprovedResNet Architecture =====")
summary(ImprovedResNet(200), input_size=(1, 3, 64, 64), device=str(device))

In [ ]:
print("\n===== ImprovedResNet Module =====")

# Use Aug_1 for training the improved ResNet model
train_tf = aug_configs['aug_1']

# Train and evaluate the Task 4 model
improve_best_acc, improve_hist, improve_best_model = train_and_evaluate(
   ImprovedResNet, {'num_classes':200}, train_tf, num_epochs=30)

print(f"ImprovedResNet best test acc: {improve_best_acc:.4f}")

# Save the Task 4 model checkpoint
torch.save(improve_best_model.state_dict(), root_dir+"improve_model.pth")

# Plot training/testing loss and accuracy curves
(train_losses, train_accs, test_losses, test_accs) = improve_hist
plot_curve(train_losses, test_losses, train_accs, test_accs)

In [ ]:
plot_curve(train_losses,test_losses,train_accs,test_accs)

#### Load improve ResNet

In [ ]:
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(channels, channels // reduction),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid()
        )
    def forward(self, x):
        scale = self.fc(x).view(x.size(0), x.size(1), 1, 1)
        return x * scale

class DropBlock(nn.Module):
    def __init__(self, block_size=7, keep_prob=0.9):
        super().__init__()
        self.block_size = block_size
        self.keep_prob = keep_prob
    def forward(self, x):
        if not self.training or self.keep_prob == 1.0:
            return x
        gamma = (1 - self.keep_prob) / (self.block_size ** 2)
        mask = (torch.rand(x.shape[0], 1, x.shape[2], x.shape[3]) < gamma).to(x.device)
        mask = mask.float()
        mask = torch.nn.functional.max_pool2d(mask, kernel_size=self.block_size, stride=1, padding=self.block_size//2)
        mask = 1 - mask
        x = x * mask
        return x * (mask.numel() / mask.sum())

def drop_path(x, drop_prob=0., training=False):
    if drop_prob == 0. or not training:
        return x
    keep_prob = 1 - drop_prob
    shape = (x.shape[0],) + (1,) * (x.ndim - 1)
    random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
    random_tensor.floor_()
    return x.div(keep_prob) * random_tensor

class DropPath(nn.Module):
    def __init__(self, drop_prob=None):
        super().__init__()
        self.drop_prob = drop_prob
    def forward(self, x):
        return drop_path(x, self.drop_prob, self.training)

class PreActResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, drop_rate=0.0, use_se=True):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_ch)
        self.relu = nn.ReLU(inplace=True)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride, 1)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1)
        self.se = SEBlock(out_ch) if use_se else nn.Identity()
        self.dropblock = DropBlock(block_size=5, keep_prob=0.9)
        self.drop_path = DropPath(drop_rate)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Conv2d(in_ch, out_ch, 1, stride, 0)
    def forward(self, x):
        identity = self.shortcut(x)
        out = self.relu(self.bn1(x))
        out = self.conv1(out)
        out = self.relu(self.bn2(out))
        out = self.conv2(out)
        out = self.se(out)
        out = self.dropblock(out)
        out = self.drop_path(out)
        out += identity
        return out

class ImprovedResNet(nn.Module):
    def __init__(self, num_classes=200, width_factor=1, stage_blocks=[2,2,2,2], use_se=True, drop_rate=0.1):
        super().__init__()
        # stem
        self.conv1 = nn.Conv2d(3, int(64*width_factor), 3, 1, 1)
        self.bn1 = nn.BatchNorm2d(int(64*width_factor))
        self.relu = nn.ReLU(inplace=True)
        # stage
        self.layer1 = self._make_layer(int(64*width_factor), int(64*width_factor), stage_blocks[0], stride=1, use_se=use_se, drop_rate=drop_rate)
        self.layer2 = self._make_layer(int(64*width_factor), int(128*width_factor), stage_blocks[1], stride=2, use_se=use_se, drop_rate=drop_rate)
        self.layer3 = self._make_layer(int(128*width_factor), int(256*width_factor), stage_blocks[2], stride=2, use_se=use_se, drop_rate=drop_rate)
        self.layer4 = self._make_layer(int(256*width_factor), int(512*width_factor), stage_blocks[3], stride=2, use_se=use_se, drop_rate=drop_rate)

        self.avgpool = nn.AdaptiveAvgPool2d((1,1))
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(int(512*width_factor), num_classes)
    def _make_layer(self, in_ch, out_ch, blocks, stride, use_se, drop_rate):
        layers = []
        layers.append(PreActResBlock(in_ch, out_ch, stride, drop_rate, use_se))
        for _ in range(1, blocks):
            layers.append(PreActResBlock(out_ch, out_ch, 1, drop_rate, use_se))
        return nn.Sequential(*layers)
    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc(x)
        return x

# Load the saved Task 4 checkpoint
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint_path = root_dir + "improve_model.pth"

state_dict = torch.load(checkpoint_path, map_location=device)

# Rebuild the ImprovedResNet structure and load the saved weights
improve_model = ImprovedResNet()
improve_model.load_state_dict(state_dict)